# 🍽️ Fine-Tuning: Asistente de Comida Chilena

Este notebook realiza el fine-tuning de **DeepSeek-R1-Distill-Qwen-1.5B** usando **Unsloth** para crear un asistente especializado en comida chilena.

### Stack utilizado
- 🤖 **Modelo base**: `unsloth/DeepSeek-R1-Distill-Qwen-1.5B`
- ⚡ **Framework**: Unsloth (2x más rápido que HuggingFace)
- 🎯 **Técnica**: LoRA / QLoRA (4-bit quantization)
- 📦 **Trainer**: TRL `SFTTrainer`

### Archivos necesarios
- `comida_chilena_train.json` — Datos de entrenamiento (234 ejemplos)
- `comida_chilena_valid.json` — Datos de validación (42 ejemplos)


## 1. Instalación de dependencias

In [ ]:
# Instalar Unsloth (versión optimizada para Colab/Jupyter con GPU)

import subprocess, sys

# Detectar versión de CUDA y torch para instalar unsloth correcto
subprocess.run([sys.executable, '-m', 'pip', 'install', 'unsloth[colab-new]', '-q'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'trl', 'peft', 'accelerate', 'bitsandbytes', '-q'])

print('✅ Instalación completada')

✅ Instalación completada


## 2. Configuración de hiperparámetros

In [ ]:
# ─────────────────────────────────────────────
#  CONFIGURACIÓN PRINCIPAL — ajusta según GPU
# ─────────────────────────────────────────────

MODEL_NAME       = "suayptalha/DeepSeek-R1-Distill-Llama-3B"  # Modelo base
MAX_SEQ_LENGTH   = 512     # Suficiente para este dataset (ejemplos cortos ~73 palabras)
DTYPE            = None    # None = autodetectar (bfloat16 en Ampere+, float16 en otros)
LOAD_IN_4BIT     = True    # QLoRA: cuantización 4-bit para reducir VRAM

# LoRA — parámetros de adaptación
LORA_R           = 16       # Rank de LoRA. Apropiado para modelo 1.5B
LORA_ALPHA       = 32      # Escala de LoRA (2x rank para mejor aprendizaje)
LORA_DROPOUT     = 0.05    # Pequeño dropout para evitar overfitting

# Entrenamiento
TRAIN_FILE       = "comida_chilena_train.json"
VALID_FILE       = "comida_chilena_valid.json"
OUTPUT_DIR       = "./comida-chilena-lora"   # Directorio donde se guardan los checkpoints
MAX_STEPS        = -1      # -1 = usar epochs en vez de steps
NUM_EPOCHS       = 5       # 5 epochs es óptimo para 234 ejemplos
BATCH_SIZE       = 2       # Por GPU. Reducir a 1 si hay OOM
GRAD_ACCUM       = 4       # Gradient accumulation (batch efectivo = BATCH_SIZE * GRAD_ACCUM = 8)
LEARNING_RATE    = 2e-4    # Más conservador que 2e-4 para modelo pequeño
WARMUP_RATIO     = 0.1     # 10% de warmup

print('✅ Configuración cargada')
print(f'   Modelo:   {MODEL_NAME}')
print(f'   Modo:     {"QLoRA 4-bit" if LOAD_IN_4BIT else "LoRA 16-bit"}')
print(f'   LoRA r:   {LORA_R} | alpha: {LORA_ALPHA}')
print(f'   Epochs:   {NUM_EPOCHS}')
print(f'   Batch efectivo: {BATCH_SIZE * GRAD_ACCUM}')


✅ Configuración cargada
   Modelo:   suayptalha/DeepSeek-R1-Distill-Llama-3B
   Modo:     QLoRA 4-bit
   LoRA r:   16 | alpha: 32
   Epochs:   5
   Batch efectivo: 8


## 3. Carga del modelo base

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
    # token = "hf_...",  # Descomenta si usas un modelo privado de HuggingFace
)

print('✅ Modelo base cargado')
print(f'   Parámetros totales: {sum(p.numel() for p in model.parameters()):,}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

Unsloth: Will load suayptalha/DeepSeek-R1-Distill-Llama-3B as a legacy tokenizer.


suayptalha/DeepSeek-R1-Distill-Llama-3B does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.
✅ Modelo base cargado
   Parámetros totales: 1,803,463,680


## 4. Aplicar LoRA al modelo

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # Atención
        "gate_proj", "up_proj", "down_proj",        # MLP / FFN
    ],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",      # No entrenar bias
    use_gradient_checkpointing = "unsloth",  # Ahorra ~30% VRAM
    random_state   = 42,
    use_rslora     = False,       # Rank-stabilized LoRA (experimental)
)

# Mostrar parámetros entrenables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA aplicado')
print(f'   Parámetros entrenables: {trainable:,} ({100*trainable/total:.2f}% del total)')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.7 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA aplicado
   Parámetros entrenables: 24,313,856 (1.33% del total)


## 5. Preparación del dataset

Los datos están en formato **conversacional** (system / user / assistant). Usamos el chat template de Llama-3.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Aplicar el chat template de Qwen-2.5 (base de DeepSeek-R1)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def format_conversations(examples):
    """Convierte la lista de mensajes al formato de texto del chat template."""
    texts = []
    for conversation in examples["conversations"]:
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize              = False,
            add_generation_prompt = False,
        )
        texts.append(text)
    return {"text": texts}

# Cargar datasets
train_dataset = load_dataset(
    "json",
    data_files = {"train": TRAIN_FILE},
    split      = "train",
)
valid_dataset = load_dataset(
    "json",
    data_files = {"validation": VALID_FILE},
    split      = "validation",
)

# Aplicar formato
train_dataset = train_dataset.map(format_conversations, batched=True)
valid_dataset = valid_dataset.map(format_conversations, batched=True)

print(f'✅ Dataset cargado')
print(f'   Ejemplos de entrenamiento: {len(train_dataset)}')
print(f'   Ejemplos de validación:    {len(valid_dataset)}')
print()
print('--- Primer ejemplo (preview) ---')
print(train_dataset[0]['text'][:500], '...')

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/234 [00:00<?, ? examples/s]

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

✅ Dataset cargado
   Ejemplos de entrenamiento: 234
   Ejemplos de validación:    42

--- Primer ejemplo (preview) ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Eres una asistente virtual especializada en comida chilena. Tu rol es responder de forma simple, clara y útil sobre platos típicos de Chile, ingredientes, preparaciones básicas, acompañamientos y consejos caseros. Usas un tono amable y directo. Si no sabes algo, lo dices con honestidad.<|eot_id|><|start_header_id|>user<|end_header_id|>

¿Qué es el sándwich chacarero?<|eot_ ...


## 6. Configurar el Trainer (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = valid_dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = True,   # True acelera entrenamiento con secuencias cortas
    args = TrainingArguments(
        per_device_train_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps  = GRAD_ACCUM,
        warmup_ratio                 = WARMUP_RATIO,
        num_train_epochs             = NUM_EPOCHS,
        max_steps                    = MAX_STEPS,
        learning_rate                = LEARNING_RATE,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 10,
        eval_strategy                = "steps",
        eval_steps                   = 50,
        save_strategy                = "steps",
        save_steps                   = 50,
        load_best_model_at_end       = True,
        metric_for_best_model        = "eval_loss",
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "cosine",   # Cosine es mejor que linear para pocos datos
        seed                         = 42,
        output_dir                   = OUTPUT_DIR,
        report_to                    = "none",
    ),
)

print('✅ Trainer configurado')
print(f'   LR scheduler: cosine')
print(f'   Epochs: {NUM_EPOCHS} | Eval cada 50 steps')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/234 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/42 [00:00<?, ? examples/s]

✅ Trainer configurado
   LR scheduler: cosine
   Epochs: 5 | Eval cada 50 steps


## 7. Entrenamiento 🚀

In [ ]:
import time

# Mostrar uso de GPU antes
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU: {gpu_stats.name} | VRAM total: {max_memory} GB | Reservado: {start_gpu_memory} GB')

# Entrenar
start_time = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start_time

# Mostrar estadísticas
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f'\n✅ Entrenamiento completado')
print(f'   Tiempo total:         {elapsed/60:.1f} minutos')
print(f'   VRAM usada:           {used_memory} GB / {max_memory} GB')
print(f'   Loss final:           {trainer_stats.training_loss:.4f}')

GPU: Tesla T4 | VRAM total: 14.563 GB | Reservado: 2.283 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 234 | Num Epochs = 5 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,0.403715,0.428396
100,0.119619,0.287368
150,0.072303,0.271549


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


✅ Entrenamiento completado
   Tiempo total:         5.5 minutos
   VRAM usada:           4.547 GB / 14.563 GB
   Loss final:           0.4255


## 8. Prueba de inferencia — ¡Habla con el Asistente! 💬


In [ ]:
# Activar modo de inferencia (2x más rápido)
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "Eres una asistente virtual especializada en comida chilena. "
    "Tu rol es responder de forma simple, clara y útil sobre platos típicos de Chile, "
    "ingredientes, preparaciones básicas, acompañamientos y consejos caseros. "
    "Usas un tono amable y directo. Si no sabes algo, lo dices con honestidad."
)

def chat(user_message: str, max_new_tokens: int = 200) -> str:
    """Envía un mensaje al asistente y devuelve su respuesta."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids      = inputs,
        max_new_tokens = max_new_tokens,
        use_cache      = True,
        temperature    = 0.3,   # Bajo para respuestas consistentes y factuales
        top_p          = 0.85,
        repetition_penalty = 1.1,
    )
    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens = True,
    )
    # Limpiar posibles bloques <think> residuales
    import re
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    return response


# ─── PRUEBAS ───
test_questions = [
    "¿Qué es el sándwich chacarero?",
    "¿Cómo se prepara el pastel de choclo?",
    "¿Cuáles son los ingredientes de la cazuela chilena?",
    "¿Qué acompañamiento va bien con las empanadas?",
    "¿Cuál es la diferencia entre porotos granados y porotos con riendas?",
]

for q in test_questions:
    print(f'👤 Usuario: {q}')
    resp = chat(q)
    print(f'🍽️  Asistente: {resp}')
    print('─' * 70)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


👤 Usuario: ¿Qué es el sándwich chacarero?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

🍽️  Asistente: El chacarero es un sándwich chileno hecho con carne, tomate, porotos verdes y ají verde. Se sirve en pan frica o marraqueta.
──────────────────────────────────────────────────────────────────────
👤 Usuario: ¿Cómo se prepara el pastel de choclo?


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🍽️  Asistente: El pastel de choclo se prepara con una base de pino o pan remojado, luego se aliña con ajo, sal y aceite. Después se coloca la mezcla de choclo (maní, leche, huevo, azúcar) y se hornea hasta dorar.
──────────────────────────────────────────────────────────────────────
👤 Usuario: ¿Cuáles son los ingredientes de la cazuela chilena?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🍽️  Asistente: La cazuela chilena suele llevar carne de vacuno o pollo, papa, zapallo, choclo, arroz o fideos, porotos verdes y verduras como zanahoria y cebolla. Es una sopa contundente y casera.
──────────────────────────────────────────────────────────────────────
👤 Usuario: ¿Qué acompañamiento va bien con las empanadas?


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🍽️  Asistente: Con las empanadas puedes llevar tomate fresco, pebre, ensalada chilena, papas cocidas o fritas, y un sándwich de pino al horno.
──────────────────────────────────────────────────────────────────────
👤 Usuario: ¿Cuál es la diferencia entre porotos granados y porotos con riendas?
🍽️  Asistente: Los porotos granados y porotos con riendas son preparaciones tradicionales con granillo de ácido fólico. La principal diferencia es que el granillo del porotos granado es más grande y blando, mientras que en el porotos con riendas es más pequeño y firme.
──────────────────────────────────────────────────────────────────────


## 9. Guardar el modelo fine-tuneado

In [ ]:
# ─── OPCIÓN A: Guardar solo los adaptadores LoRA (liviano, ~50-200 MB) ───
model.save_pretrained("comida-chilena-lora-adapters")
tokenizer.save_pretrained("comida-chilena-lora-adapters")
print('✅ Adaptadores LoRA guardados en ./comida-chilena-lora-adapters')


Unsloth: Restored added_tokens_decoder metadata in comida-chilena-lora-adapters/tokenizer_config.json.


✅ Adaptadores LoRA guardados en ./comida-chilena-lora-adapters


In [ ]:
# ─── OPCIÓN B: Guardar modelo completo fusionado (16-bit) ───
model.save_pretrained_merged(
    "comida-chilena-merged-16bit",
    tokenizer,
    save_method = "merged_16bit",
)
print('✅ Modelo fusionado (16-bit) guardado en ./comida-chilena-merged-16bit')


Unsloth: Restored added_tokens_decoder metadata in comida-chilena-merged-16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `comida-chilena-merged-16bit`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `comida-chilena-merged-16bit`:  50%|█████     | 1/2 [01:15<01:15, 75.51s/it]
Unsloth: Copying 2 files from cache to `comida-chilena-merged-16bit`: 100%|██████████| 2/2 [01:37<00:00, 48.62s/it]


Successfully copied all 2 files from cache to `comida-chilena-merged-16bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 15857.48it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:07<00:00, 63.56s/it]


Unsloth: Merge process complete. Saved to `/content/comida-chilena-merged-16bit`
✅ Modelo fusionado (16-bit) guardado en ./comida-chilena-merged-16bit


In [ ]:
# ─── OPCIÓN C: Exportar a GGUF para usar con Ollama ───
# Q4_K_M = mejor balance calidad/tamaño para Ollama
model.save_pretrained_gguf(
    "comida-chilena-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)
print('✅ Modelo GGUF exportado en ./comida-chilena-gguf/')
print()
print('Para usarlo en Ollama:')
print('  ollama create comida-chilena -f ./comida-chilena-gguf/Modelfile')
print('  ollama run comida-chilena')

# Descargar el archivo GGUF a tu PC
from google.colab import files
import glob
gguf_files = glob.glob('comida-chilena-gguf/*.gguf')
if gguf_files:
    print(f'\n⬇️  Descargando {gguf_files[0]}...')
    files.download(gguf_files[0])
else:
    print('⚠️  No se encontró archivo .gguf, revisa la carpeta manualmente')


Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in comida-chilena-gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `comida-chilena-gguf`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `comida-chilena-gguf`:  50%|█████     | 1/2 [00:58<00:58, 58.58s/it]
Unsloth: Copying 2 files from cache to `comida-chilena-gguf`: 100%|██████████| 2/2 [01:20<00:00, 40.28s/it]


Successfully copied all 2 files from cache to `comida-chilena-gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 11983.73it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:34<00:00, 47.16s/it]


Unsloth: Merge process complete. Saved to `/content/comida-chilena-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes


In [ ]:
# ─── OPCIÓN D: Subir a HuggingFace Hub ───
# Descomenta y reemplaza con tu token y nombre de repositorio

# HF_TOKEN = "hf_..."
# REPO_NAME = "tu-usuario/uls-minerva-lora"
# model.push_to_hub(REPO_NAME, token=HF_TOKEN)
# tokenizer.push_to_hub(REPO_NAME, token=HF_TOKEN)
# print(f'✅ Modelo subido a https://huggingface.co/{REPO_NAME}')

## 10. Consejos para mejorar el modelo

### 📊 Más datos = mejor modelo
Con 234 ejemplos el modelo aprende bien el dominio, pero puedes mejorarlo agregando:
- Recetas completas con pasos detallados
- Variaciones regionales de platos chilenos
- Preguntas sobre ingredientes difíciles de conseguir y sustitutos
- Consejos de cocina y errores comunes

### 🔧 Parámetros a ajustar si agregas más datos
| Parámetro | Dataset actual (234) | Dataset mediano (500+) | Dataset grande (1000+) |
|-----------|---------------------|------------------------|------------------------|
| `NUM_EPOCHS` | 5 | 3-4 | 2-3 |
| `LORA_R` | 8 | 16 | 32 |
| `LEARNING_RATE` | 1e-4 | 2e-4 | 1e-4 |

### ⚠️ Señales de overfitting
Si ves que `train_loss` baja pero `eval_loss` sube → el modelo está memorizando.
Solución: bajar epochs o activar `load_best_model_at_end = True` (ya está activado).

### 🚀 Modelfile para Ollama
```
FROM ./comida-chilena-gguf-Q4_K_M.gguf

PARAMETER temperature 0.3
PARAMETER top_p 0.85
PARAMETER repeat_penalty 1.1
PARAMETER num_predict 200
PARAMETER num_ctx 2048

PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|start_header_id|>"
PARAMETER stop "<|end_header_id|>"
PARAMETER stop "</think>"

TEMPLATE """{{ if .System }}<|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|>{{ end }}<|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>"""

SYSTEM """Eres una asistente virtual especializada en comida chilena..."""
```